In [1]:
import pandas as pd

print("Loading WSB dataset...")
# 1. Force UTF-8 encoding to save the emojis
wsb_df = pd.read_csv('../assets/reddit_wsb.csv', encoding='utf-8')

# 2. Fill empty body cells with blank strings, then combine with the title
wsb_df['body'] = wsb_df['body'].fillna('')
wsb_df['full_text'] = wsb_df['title'] + " " + wsb_df['body']

# 3. Convert the string timestamp to a clean Date object
wsb_df['timestamp'] = pd.to_datetime(wsb_df['timestamp'])
wsb_df['Date'] = wsb_df['timestamp'].dt.date

# 4. Keep only the columns we actually need for the ML model
clean_wsb = wsb_df[['Date', 'full_text', 'score']]

print(f"Cleaned data shape: {clean_wsb.shape}")
clean_wsb.head()

Loading WSB dataset...
Cleaned data shape: (53187, 3)


,Date,full_text,score
0,2021-01-28,"It's not about the money, it's about sending a...",55
1,2021-01-28,Math Professor Scott Steiner says the numbers ...,110
2,2021-01-28,Exit the system The CEO of NASDAQ pushed to ha...,0
3,2021-01-28,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,29
4,2021-01-28,"Not to distract from GME, just thought our AMC...",71


In [3]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import yfinance as yf

# 1. Download the VADER lexicon (only needs to run once)
nltk.download('vader_lexicon')
analyzer = SentimentIntensityAnalyzer()

print("Calculating sentiment for 53k rows... (this might take a minute or two)")

# 2. Function to get the compound sentiment score (-1 to 1)
def get_sentiment(text):
    return analyzer.polarity_scores(str(text))['compound']

# Apply the NLP model to our text
clean_wsb['sentiment'] = clean_wsb['full_text'].apply(get_sentiment)

# 3. Group by Date to get the Daily Average Emotion Score
# We will also sum the 'score' (upvotes) to measure total daily engagement/hype volume
daily_sentiment = clean_wsb.groupby('Date').agg(
    Daily_Emotion_Score=('sentiment', 'mean'),
    Total_Hype_Volume=('score', 'sum'),
    Post_Count=('sentiment', 'count')
).reset_index()

# Convert Date to datetime for merging
daily_sentiment['Date'] = pd.to_datetime(daily_sentiment['Date'])

# 4. Fetch matching S&P 500 data
start_date = daily_sentiment['Date'].min()
end_date = daily_sentiment['Date'].max()

print(f"\nFetching S&P 500 data from {start_date.date()} to {end_date.date()}...")
sp500 = yf.download('^GSPC', start=start_date, end=end_date)
sp500 = sp500[['Close', 'Volume']].reset_index()
sp500['Date'] = pd.to_datetime(sp500['Date']).dt.tz_localize(None)
sp500.columns = ['Date', 'SP500_Close', 'SP500_Volume']

# 5. THE FINAL MERGE
final_dataset = pd.merge(daily_sentiment, sp500, on='Date', how='inner')

# Save our golden dataset to the assets folder!
final_dataset.to_csv('../assets/final_training_data.csv', index=False)

print(f"\nSuccess! Final dataset shape: {final_dataset.shape}")
final_dataset.head()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...


Calculating sentiment for 53k rows... (this might take a minute or two)

Fetching S&P 500 data from 2020-09-29 to 2021-08-16...


[*********************100%***********************]  1 of 1 completed


Success! Final dataset shape: (122, 6)


,Date,Daily_Emotion_Score,Total_Hype_Volume,Post_Count,SP500_Close,SP500_Volume
0,2020-09-29,0.912700,4,1,3335.469971,3661590000
1,2021-01-28,0.023345,1149849,1197,3787.379883,6992770000
2,2021-01-29,-0.008826,6410329,15694,3714.239990,6643370000
3,2021-02-01,0.169128,3739587,884,3773.860107,5436230000
4,2021-02-02,0.100040,1180580,1502,3826.310059,5514090000
